# Setup

In [1]:
!wget -q https://raw.githubusercontent.com/PhilBurub/Evaluating-GAN-and-VAE-for-Image-Captioning/refs/heads/main/code/utils.py
!wget -q https://raw.githubusercontent.com/PhilBurub/Evaluating-GAN-and-VAE-for-Image-Captioning/refs/heads/main/code/architectures/adapter.py
!wget -q https://raw.githubusercontent.com/PhilBurub/Evaluating-GAN-and-VAE-for-Image-Captioning/refs/heads/main/code/architectures/lm-decoding/baseline_trainer.py
!wget -q https://raw.githubusercontent.com/PhilBurub/Evaluating-GAN-and-VAE-for-Image-Captioning/refs/heads/main/data/ds_captions.json

In [2]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoConfig,
    AutoImageProcessor,
    AutoModel
)
from PIL import Image
from adapter import ImageConvAdapter
from baseline_trainer import QwenImageDescriptionTrainer
from torch.utils.data import DataLoader
from utils import (
    _set_env,
    embed,
    get_collator
)
from torchvision.io import read_image
import pandas as pd
import torch
import wandb
import json

_set_env("WANDB_API_KEY")

2026-05-02 19:40:09.682079: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777750809.869448      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777750809.923539      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777750810.352346      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777750810.352391      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777750810.352394      57 computation_placer.cc:177] computation placer alr

WANDB_API_KEY:  ········


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


# Data

In [4]:
images_path = '/kaggle/input/datasets/adityajn105/flickr30k/Images/flickr30k_images/'
captions = pd.read_json('ds_captions.json', orient='records')
captions.split.value_counts()

split
train    22220
test      6350
val       3174
Name: count, dtype: int64

# Image Model

In [5]:
image_model_name = 'facebook/dinov2-base'
image_processor = AutoImageProcessor.from_pretrained(image_model_name, use_fast=True)
image_model = AutoModel.from_pretrained(image_model_name).to(device)

image_emb_sample = embed(
    [read_image(images_path + image) for image in captions.head().image],
    image_model,
    image_processor,
    device
)

preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

# Language Model

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer

qwen_model_name = "Qwen/Qwen3-0.6B"
qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_model_name)
qwen_model = AutoModelForCausalLM.from_pretrained(qwen_model_name).to(device)

qwen_hidden_size = qwen_model.config.hidden_size
image_model_hidden_size = image_emb_sample.shape[-1]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

# Dataloaders

In [7]:
collate = get_collator(
    image_model,
    image_processor,
    qwen_tokenizer.special_tokens_map['eos_token'],
    device,
    images_path
)

test_collate = get_collator(
    image_model,
    image_processor,
    '',
    device,
    images_path
)

In [8]:
train_sample_loader = DataLoader(
    captions[captions.split == 'train'].head(2000).to_dict(orient='records'),
    batch_size=16,
    shuffle=True,
    collate_fn=collate
)

train_loader = DataLoader(
    captions[captions.split == 'train'].to_dict(orient='records'),
    batch_size=16,
    shuffle=True,
    collate_fn=collate
)

val_loader = DataLoader(
    captions[captions.split == 'val'].to_dict(orient='records'),
    batch_size=16,
    shuffle=False,
    collate_fn=collate
)

test_loader = DataLoader(
    captions[captions.split == 'test'].to_dict(orient='records'),
    batch_size=16,
    shuffle=False,
    collate_fn=test_collate
)

# Adapter Initialization

In [9]:
adapter_config = {
    'input_dim': image_model_hidden_size,
    'output_dim': qwen_hidden_size,
    'hidden_dim': 768,
    'kernel_size': 5,
    'kernel_stride': 4,
    'layers_num': 2
}

image_adapter = ImageConvAdapter(**adapter_config).to(device)
print("Trainable parameters in adapter:", sum(p.numel() for p in image_adapter.parameters() if p.requires_grad))

Trainable parameters in adapter: 6883072


In [10]:
for image_inputs, _ in train_sample_loader:
    break
image_adapter(image_inputs.permute((0, 2, 1))).shape

torch.Size([16, 1024, 15])

# lr Optimization

In [11]:
best_lr = None
best_loss = None

for lr in (1e-6, 5e-6, 1e-5, 5e-5, 1e-4, 5e-4, 1e-3, 5e-3, 1e-2):
    image_adapter = ImageConvAdapter(**adapter_config).to(device)
    
    trainer = QwenImageDescriptionTrainer(
        qwen_model,
        qwen_tokenizer,
        image_adapter,
        lr=lr,
        device=device
    )

    trainer.run_epoch(train_sample_loader)
    loss = trainer.run_epoch(val_loader, val=True)
    
    print('lr: %f; loss: %.4f'%(lr, loss))
    if best_loss is None or loss < best_loss:
        best_lr, best_loss = lr, loss

best_lr

Validating: 100%|██████████| 199/199 [01:42<00:00,  1.95it/s]


lr: 0.000001; loss: 7.1441


Validating: 100%|██████████| 199/199 [01:15<00:00,  2.62it/s]


lr: 0.000005; loss: 6.3336


Validating: 100%|██████████| 199/199 [01:16<00:00,  2.62it/s]


...

Validating: 100%|██████████| 199/199 [01:15<00:00,  2.63it/s]


lr: 0.005000; loss: 4.8295


Validating: 100%|██████████| 199/199 [01:15<00:00,  2.63it/s]


lr: 0.010000; loss: 5.5436


0.0005

# Training Loop

In [12]:
image_adapter = ImageConvAdapter(**adapter_config).to(device)
trainer = QwenImageDescriptionTrainer(
    qwen_model,
    qwen_tokenizer,
    image_adapter,
    lr=best_lr,
    device=device
)

In [13]:
epoch_num = 7
config = {
    'architecture': 'baseline',
    'lr': best_lr,
    'adapter_config': adapter_config
}

with wandb.init(
    entity="pburub",
    project="gan-vae-image-captioning",
    config=config
) as run:
    for i in range(epoch_num):
        train_loss = trainer.run_epoch(train_loader)
        val_loss = trainer.run_epoch(val_loader, val=True)
        torch.save(image_adapter, str(i) + '_checkpoint.model')

        run.log({
            "train_loss": train_loss,
            "val_loss": val_loss,
            "epoch": i
        })

        model_artifact = wandb.Artifact(
            name=str(i) + "_checkpoint",
            type="model"
        )
        model_artifact.add_file(str(i) + '_checkpoint.model')
        run.log_artifact(model_artifact)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Currently logged in as: pburub to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Validating: 100%|██████████| 199/199 [01:18<00:00,  2.54it/s]


epoch,▁▂▃▅▆▇█
train_loss,█▆▅▄▃▂▁
val_loss,▄▁▁▂▃▆█
epoch,6
train_loss,1.72942
val_loss,2.994
